# Lab: Machine Learning for Anomalous Property Classification — Part 2

### Preamble

In Part 1, you built a classifier for the National Anomalous Property Register by applying each preprocessing step manually and in sequence: encoding categorical columns, assembling feature vectors, normalising, fitting a model. The result worked, but the steps were scattered across cells and tightly coupled to a particular DataFrame. Running the same preprocessing on new data — say, a fresh batch of field reports from the regional offices — would mean re-executing every cell in the right order and hoping nothing had changed in scope.

Part 2 introduces the **ML Pipeline**: a single object that encapsulates the entire preprocessing and training workflow. Fitting a pipeline on training data produces a `PipelineModel` that can be saved, loaded, and applied to new data with a single `.transform()` call. We also introduce **Word2Vec** for embedding the free-text columns, and **CrossValidator** for systematic hyperparameter search — both of which slot naturally into a pipeline.

HMSPIA has noted the improvement. The note is filed under *Technical Progress* and will be reviewed when the relevant committee reconvenes.

## 1. Setup

In [ ]:
!pip install pyspark matplotlib scikit-learn "nbformat>=4.2.0" --quiet

In [ ]:
import math
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, size, concat_ws, split
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder, VectorAssembler, 
                                Imputer, MinMaxScaler, Word2Vec, Tokenizer, PCA)
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.stat import Summarizer

spark = SparkSession.builder.getOrCreate()


## 2. Loading the Dataset

We load the same NAPR Parquet file from Part 1 and apply the same light preprocessing — null drop, derived features, boolean and target casting — in a single consolidated block. This is boilerplate you have already seen; run it and move on.

In [ ]:
properties = spark.read.parquet("haunted_houses.parquet")
properties.count(), len(properties.columns)

In [ ]:
# Null drop, derived features, cast target — identical to Part 1
properties = properties.na.drop()

CURRENT_YEAR = 2026
properties = properties \
    .withColumn("property_age", lit(CURRENT_YEAR) - col("construction_date")) \
    .withColumn("sightings_count", size(split(col("sightings"), r"\|")))

bool_cols = ["warning", "teenage_group_disappearance", "foggy", "belltower", "moor_nearby"]
for c in bool_cols:
    properties = properties.withColumn(c, col(c).cast(IntegerType()))
properties = properties.withColumn("haunted", col("haunted").cast(IntegerType()))

TARGET = "haunted"

properties.count()  # records remaining after null drop


## 3. Transformers and Estimators

Before building a pipeline, it helps to be precise about the two kinds of objects it can contain.

**Transformer** — has only a `transform(df)` method. It applies a fixed, data-independent operation: `VectorAssembler`, `OneHotEncoder` (after fitting), `Tokenizer`. No learning happens; the same logic applies to any DataFrame.

**Estimator** — has a `fit(df)` method that *learns* something from the data and returns a fitted **transformer** (a `Model`). Examples: `StringIndexer`, `Imputer`, `MinMaxScaler`, `Word2Vec`, `LogisticRegression`. You must call `.fit()` before you can call `.transform()`.

The distinction matters because a `Pipeline` handles them differently under the hood: for each stage it calls `fit()` if the stage is an estimator, `transform()` if it is a transformer, passing the output of one stage as the input to the next.

### Data contamination

Estimators *learn from whatever data you pass to `.fit()`*. If you fit an `Imputer` or `MinMaxScaler` on the full dataset before splitting, the learned statistics (mean, min, max) will reflect test-set values too — the model has indirectly 'seen' the test set before evaluation. This is **data leakage**, and it produces optimistically biased metrics.

The correct discipline is: **fit only on training data, transform both splits**.

```python
# Wrong — leaks test statistics into the fitted parameters
imputer_model = imputer.fit(full_dataset)

# Correct — learns only from training data
imputer_model = imputer.fit(training)
train_imputed = imputer_model.transform(training)
test_imputed  = imputer_model.transform(test)
```

### Is there a `fit_transform()` like in scikit-learn?

No. Spark ML does not have `fit_transform()`, and the `Pipeline` object is the idiomatic replacement. Calling `pipeline.fit(training)` fits every estimator stage on the training data, producing a `PipelineModel`. That model is then a pure transformer: `pipeline_model.transform(test)` applies all stages — with the parameters learned from training — to the test set without refitting anything.

## 4. Building a Pipeline

A `Pipeline` is itself an estimator whose only parameter is `stages`: an ordered list of transformers and estimators. Calling `.fit()` on a Pipeline produces a `PipelineModel`, which is a transformer. That `PipelineModel` can then be applied to new data with a single `.transform()` call — the entire preprocessing chain runs automatically in the correct order.

This is the key practical advantage over Part 1's manual approach: the pipeline guarantees that test data and any future production data go through exactly the same transformations as the training data, fitted on the training data only.

### 4.1 Preprocessing stages

We define each preprocessing stage as a named variable. The boilerplate (column lists, output column names) is filled in; fill in the missing estimator and transformer definitions.

**TODO:**
- Instantiate `StringIndexer` for `phenomenon_type` and `ghost`.
- Instantiate `OneHotEncoder` on the resulting index columns.
- Instantiate `Imputer` with `strategy="mean"` on the numeric columns.
- Instantiate `MinMaxScaler` to bring the assembled numeric vector to [0, 1].
- Instantiate the final `VectorAssembler` that combines scaled numerics, boolean flags, and OHE vectors into `"features"`.

In [ ]:
# --- Categorical encoding ---
indexer = StringIndexer(
    inputCols     = ["phenomenon_type", "ghost"],
    outputCols    = ["phenomenon_idx", "ghost_idx"],
    handleInvalid = "keep",
)

encoder = OneHotEncoder(
    inputCols  = ["phenomenon_idx", "ghost_idx"],
    outputCols = ["phenomenon_vec", "ghost_vec"],
)

# --- Numeric imputation ---
numeric_cols = [
    "property_age", "sightings_count",
    "owl_hoots", "villagers_count", "disappeared_visitors",
    "dread", "tripadvisor_stars", "insurance_exclusion_clauses",
]
numeric_cols_imputed = [c + "_i" for c in numeric_cols]

imputer = Imputer(
    strategy   = "mean",
    inputCols  = numeric_cols,
    outputCols = numeric_cols_imputed,
)

# --- Scaling: bring all numeric features to [0, 1] ---
numeric_assembler = VectorAssembler(inputCols=numeric_cols_imputed, outputCol="numeric_raw")

scaler = MinMaxScaler(inputCol="numeric_raw", outputCol="numeric_scaled")

# --- Final feature assembly ---
feature_assembler = VectorAssembler(
    inputCols = ["numeric_scaled"] + bool_cols + ["phenomenon_vec", "ghost_vec"],
    outputCol = "features",
)

### 4.2 Assembling and fitting the pipeline

We pass the stages in order to `Pipeline`, then fit on the training set. Note that we split *before* fitting — the pipeline must never see the test set during training, including during the `Imputer` and `MinMaxScaler` fitting steps.

In [ ]:
training, test = [df.cache() for df in properties.randomSplit([0.7, 0.3], seed=753)]
training.count(); test.count()  # trigger cache materialisation

In [ ]:
# Concrete example: VectorAssembler is a transformer — no fit() needed
demo_assembler = VectorAssembler(
    inputCols=["owl_hoots", "dread", "tripadvisor_stars"],
    outputCol="demo_features",
)
demo_assembler.explainParam("outputCol")

# Imputer is an estimator — it must be fitted, and only on training data.
# Fitting on the full dataset would leak test-set statistics. Note that
# the training DataFrame is defined in Section 4 — run that cell first
# if you want to execute this demo independently.
demo_imputer = Imputer(
    strategy   = "mean",
    inputCols  = ["owl_hoots", "dread"],
    outputCols = ["owl_hoots_i", "dread_i"],
)
demo_imputer_model = demo_imputer.fit(training)

# Show what the imputer learned: the mean of each column on the training set.
# These are the values that will fill nulls at transform time.
training.agg(
    {"owl_hoots": "mean", "dread": "mean"}
).show()


In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol=TARGET)

pipeline = Pipeline(stages=[
    indexer,
    encoder,
    imputer,
    numeric_assembler,
    scaler,
    feature_assembler,
    lr,
])

pipeline_model = pipeline.fit(training)

In [ ]:
results = pipeline_model.transform(test)
results.select("prediction", "probability", TARGET).show(5, truncate=50)

### 4.3 Evaluating the pipeline model

We can extract individual fitted stages from the `PipelineModel` using `.stages`, which mirrors the order of stages passed to the pipeline.

In [ ]:
evaluator = BinaryClassificationEvaluator(labelCol=TARGET, metricName="areaUnderROC")
auc_pipeline = evaluator.evaluate(results)
print(f"Pipeline LR — Test AUC-ROC: {auc_pipeline:.4f}")

# Extract the fitted LR model from the last stage
lr_model = pipeline_model.stages[-1]
metrics  = lr_model.evaluate(results.select(TARGET, "features"))
print(f"Precision (haunted=1): {metrics.precisionByLabel[1]:.4f}")
print(f"Recall    (haunted=1): {metrics.recallByLabel[1]:.4f}")

In [ ]:
# Confusion matrix
results.groupBy(TARGET).pivot("prediction").count().show()

## 5. Saving and Loading a Pipeline

A fitted `PipelineModel` can be written to disk and reloaded later. This is how a trained model moves from a training notebook to a production scoring job — the loaded model is a fully functional transformer that applies all stages to new data with a single `.transform()` call, with no retraining.

In [ ]:
pipeline_model.write().overwrite().save("./hmspia_pipeline_lr")

from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("./hmspia_pipeline_lr")
auc_loaded = evaluator.evaluate(loaded_model.transform(test))
print(f"AUC from loaded pipeline: {auc_loaded:.4f}  (should match {auc_pipeline:.4f})")

## 6. Hyperparameter Tuning with CrossValidator

`CrossValidator` wraps a pipeline (or any estimator) and a grid of hyperparameter values, refitting the pipeline once for each combination across K folds of the training data. It returns the best model according to the evaluator metric.

`ParamGridBuilder` constructs the grid. Each `.addGrid(param, values)` call adds one hyperparameter with a list of candidate values; the builder takes the Cartesian product across all added parameters.

Note: Spark ML has no random grid search equivalent. `ParamGridBuilder` always evaluates the full Cartesian product.

In [ ]:
# TODO: build a pipeline without the LR stage (preprocessing only),
# then add LR as the final stage. Define a small param grid over
# lr.regParam and lr.elasticNetParam, wrap it in CrossValidator with
# numFolds=3, fit on training, and print the best AUC.

preprocess_pipeline = Pipeline(stages=[
    indexer, encoder, imputer, numeric_assembler, scaler, feature_assembler
])

lr_crossval = LogisticRegression(featuresCol="features", labelCol=TARGET)

crossval_pipeline = Pipeline(stages=[preprocess_pipeline, lr_crossval])

# TODO: define param_grid using ParamGridBuilder
param_grid = (
#...
)

# TODO: define param_grid using CrossValidator
cross_validator = CrossValidator(
#...
)

cross_validator_model = cross_validator.fit(training)
crossval_results  = cross_validator_model.transform(test)
auc_cv      = evaluator.evaluate(crossval_results)

print(f"Cross-validated AUC-ROC: {auc_cv:.4f}")
print(f"Avg metrics per fold:    {[round(m, 4) for m in cross_validator_model.avgMetrics]}")


In [ ]:
# The best model is accessible directly
best_pipeline = cross_validator_model.bestModel
best_lr       = best_pipeline.stages[-1]   # LR is last stage of inner pipeline
print(f"Best regParam:        {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

## 7. Word2Vec for Text Features

In Part 1 Section 8.3 we used TF-IDF to incorporate free-text columns. TF-IDF is a bag-of-words approach: it counts words but ignores their meaning and order. **Word2Vec** is a shallow neural network that instead learns a dense vector representation (an *embedding*) for each word such that semantically similar words sit close together in vector space.

Spark ML's `Word2Vec` takes a column of token arrays and returns, for each document, the **average** of its word vectors — a single dense vector regardless of document length. This is cruder than transformer-based embeddings but fast and entirely Spark-native.

Key parameters:
- `vectorSize` — dimensionality of each word embedding (default 100).
- `minCount` — words appearing fewer than this many times are ignored.
- `windowSize` — context window used during training.
- `inputCol` — must be an array of strings (i.e. already tokenized).
- `outputCol` — the resulting dense vector column.

### 7.1 Preparing text columns

Word2Vec expects tokenised input — an array of strings per row. We use `Tokenizer` (lowercases and splits on whitespace) and then concatenate all four text columns into a single token array before embedding.

In [ ]:

# These are the free-text columns of our dataset
TEXT_COLS = [
    "estate_agent_euphemism", "cat_behaviour",
    "vicar_response", "most_recent_excuse",
    "previous_owners_fate", "archaeology_report_status"
]

# Fill nulls so Tokenizer doesn't fail, then concatenate into one string
data_text = properties
for text_col in TEXT_COLS:
    data_text = data_text.fillna("", subset=[text_col])

data_text = data_text.withColumn(
    "all_text",
    concat_ws(" ", *[col(text_col) for text_col in TEXT_COLS])
)

tokenizer = Tokenizer(inputCol="all_text", outputCol="text_tokens")
data_text = tokenizer.transform(data_text)
data_text.select("text_tokens").show(3, truncate=80)

### 7.2 Fitting Word2Vec as a standalone estimator

Before plugging Word2Vec into a pipeline, we inspect it in isolation to understand what it learns. After fitting we can query the model for the words most similar to a given word — a useful sanity check.

In [ ]:
# Fitting Word2Vec (vectorSize: 150, minCount: 3, windowSize: 5)
word2vec = Word2Vec(
    #...
    inputCol   = "text_tokens",
    outputCol  = "text_embedding",
    seed       = 753,
)

word2vec_model = word2vec.fit(data_text)

# TODO: Inspect the learned vocabulary — find words similar to 'vicar'

In [ ]:
# Transform: each document becomes the mean of its word vectors
data_text = word2vec_model.transform(data_text)
data_text.select("text_embedding").show(3, truncate=80)

### 7.2.1 Which text column matters most?

Before building the full pipeline, it is worth asking whether all four text columns carry equal predictive weight, or whether one dominates.

**The approach.** For each column *in isolation* we:
1. Tokenise it.
2. Fit a Word2Vec model on the training split of that column alone.
3. Transform both splits to get a document-level embedding (the mean of its token vectors).
4. Assemble that single embedding into `features` and train a Logistic Regression.
5. Record the AUC on the test split.

**What this is and isn't.** This is a *marginal contribution* screen — each column is tested in isolation, so interaction effects are ignored. It is a fast, interpretable guide for deciding which columns to invest in further (e.g. with a larger `vectorSize`, more preprocessing, or a dedicated PCA stage). It does **not** tell you the column's contribution inside a joint model, where a weaker column may still add signal that the others lack.

**TODO:** Complete the loop. For each `text_col` in `TEXT_COLS`:
1. Tokenise it with `Tokenizer` into `"single_column_tokens"`.
2. Fit `Word2Vec` (vectorSize=150, minCount=2, seed=753) on the training split.
3. Transform both splits; assemble the embedding into `"features"` with `VectorAssembler`.
4. Fit `LogisticRegression` and evaluate AUC on the test split; store it in `column_auc_scores`.

Which column wins? Does the result match your intuition about the dataset?

In [ ]:
auc_scores_by_column = {}

for text_col in TEXT_COLS:
    column_data = properties.fillna("", subset=[text_col])

    # TODO: tokenise text_col into "single_column_tokens" using Tokenizer;
    #       reassign the result to column_data
    # YOUR CODE HERE

    train_data, test_data = [df.cache() for df in column_data.randomSplit([0.7, 0.3], seed=753)]
    train_data.count(); test_data.count()

    # TODO: fit Word2Vec (vectorSize=150, minCount=2, seed=753) on train_data;
    #       transform both train_data and test_data into train_embedded / test_embedded
    # YOUR CODE HERE

    # TODO: use VectorAssembler to assemble "column_embedding" into "features";
    #       fit LogisticRegression on train_embedded; evaluate AUC on test_embedded;
    #       store the result as auc_scores_by_column[text_col]
    # YOUR CODE HERE

    train_data.unpersist(); test_data.unpersist()

for text_col, auc_score in sorted(auc_scores_by_column.items(), key=lambda x: x[1], reverse=True):
    print(f"{text_col:35s}  AUC = {auc_score:.4f}")


### 7.2.2 Vector arithmetic: exploring the embedding space

Word2Vec encodes semantic relationships as **directions** in vector space. The canonical demonstration is an analogy task: if the model has learned a consistent direction for a relationship, arithmetic on vectors should land near the expected answer.

A well-known example from generic corpora is the *country → capital* relationship:

> `vec(France) − vec(Paris) ≈ vec(Germany) − vec(Berlin)`  
> equivalently: `vec(Germany) − vec(Berlin) + vec(Paris) ≈ vec(France)`

Our corpus has its own consistent relationships across four distinct registers — the estate agent's euphemisms, the vicar's responses, the cat's behaviour, and the excuses offered by previous occupants. Good analogies exploit the *contrast between registers*:

| Column | Analogy | Intuition |
|---|---|---|
| Estate agent euphemism | `poltergeist − haunted + potential ≈ ?` | What does an agent call a poltergeist? |
| Owner Excuse | `ghost − haunted + pipes ≈ ?` | The same phenomenon, rationalised as a plumbing issue |
| Cat behaviour | `cat − ghost + interest ≈ ?` | If not a ghost, what is the resident cat then interested in? |

The arithmetic works as follows: given `positive` words (add) and `negative` words (subtract), compute a result vector, then call `findSynonyms` with it — `findSynonyms` accepts either a word **string** *or* a raw `DenseVector`.

> **Note on expectations.** The corpus is not enormous, so not every analogy will resolve cleanly — that is a valid and interesting result in itself. If a query word is missing from the vocabulary, `analogy()` will tell you. Try browsing `vocab.keys()` to find words that are actually present before designing your own experiments.

**TODO:**
1. Run at least **three** analogy experiments using `analogy()`. Start with the examples in the table above, then design one of your own.
2. Use `cosine_sim()` to evaluate at least **four** word pairs. Which are closest? Are any results surprising?


In [ ]:
from pyspark.sql.functions import udf
from pyspark.ml.linalg import VectorUDT, Vectors

vocabulary_df = word2vec_model.getVectors()

def get_vector(word):
    """Return the embedding for a single word from vocabulary_df as a DenseVector."""
    # TODO: Use vocabulary_df.filter to find the row where 'word' matches.
    # Then return the contents of the 'vector' column.
    pass

# Define a UDF to multiply a vector by a scalar
multiply_vector_by_scalar = udf(
    lambda vector, scalar: Vectors.dense([scalar * x for x in vector.toArray()]),
    VectorUDT()
)

def analogy(positive, negative=None, k=5):
    """
    Compute a query vector as: sum(pos_vectors) - sum(neg_vectors).
    
    Detailed Steps:
    1. Create a weights list of (word, weight) tuples. 
       Positive words get weight 1.0, negative words get -1.0.
    2. Convert this list to a Spark DataFrame (weights_df).
    3. Join weights_df with vocabulary_df on 'word'.
    4. Use the provided UDF to first assign weight the vectors and perform the final sum
    5. Pass that vector to word2vec_model.findSynonyms(vector, count).
    6. Filter the results to remove any words that were in the 'positive' or 'negative' lists.
    """
    negative = negative or []
    # TODO: Implement weights_df, join, Summarizer, and findSynonyms.
    pass


In [ ]:
# Experiment B - supernatural phenomenon -> mundane excuse
# The same event, rationalised away: does subtracting "haunted"
# and adding "pipes" shift the neighbourhood toward excuse vocabulary?
# TODO: call analogy() with the appropriate positive and negative words.
# YOUR CODE HERE


In [ ]:
# Experiment C - feline alarm -> feline curiosity
# Cats react to ghosts. What does a cat investigate when there is no ghost?
# TODO: call analogy() with the appropriate positive and negative words.
# YOUR CODE HERE


In [ ]:
# Experiment D - design your own analogy
# Pick two registers from the dataset and define a shift that makes semantic sense.
# TODO: call analogy() with your chosen positive and negative words.
# YOUR CODE HERE


In [ ]:
# Pairwise cosine similarities — probing cross-register distances
# TODO: for at least four word pairs, call cosine_sim() and print the result.
#   Suggested pairs: ("haunted", "potential"), ("ghost", "phenomenon"), ("hissing", "purring")
#   Which pair is closest? Are any results surprising?
# YOUR CODE HERE

### 7.2.3 Clustering the vocabulary with KMeans

Vector arithmetic probes *specific* relationships. Clustering takes a broader view: it partitions the entire vocabulary into groups of semantically close words, without any prior hypothesis about which words belong together.

Spark ML's `KMeans` is an estimator that works on any `DenseVector` column — the same interface as the rest of the pipeline. We fit it directly on `vectors_df` (the vocabulary DataFrame returned by `getVectors()`).

The key design choice is **k** — the number of clusters. Use the **silhouette score** (`ClusteringEvaluator` with `metricName="silhouette"`, `distanceMeasure="cosine"`) to compare candidate values: +1 is perfect separation, 0 means overlapping, −1 means misclustered.

To inspect a fitted cluster, a useful summary is the **words closest to its centroid** — those are the most representative members. `cluster_label()` below does this: given a cluster id and a fitted model, it computes cosine distances from all member words to the centroid and returns the top-n closest.

**TODO:**
1. Complete `cluster_label()` — the centroid retrieval and distance computation are stubbed out.
2. Sweep `k ∈ {5, 8, 12, 16, 20}`, record silhouette scores, and identify the best k.
3. Refit with the best k and call `cluster_label()` on every cluster. Can you assign a human-readable theme to each?

Consider: what makes a cluster thematically incoherent? How does increasing k trade interpretability for granularity?


In [ ]:
clustering_df = vocabulary_df.withColumnRenamed("vector", "features")

def cluster_label(cluster_id, model, data, n=8):
    """
    Identify the top n words that best represent a cluster (those closest to its centroid).
    
    Detailed Steps:
    1. Extract the centroid: model.clusterCenters()[cluster_id].
    2. Filter the 'data' DataFrame for rows where 'prediction' matches 'cluster_id'.
    3. For each member, calculate cosine similarity between its 'features' and the 'centroid'.
       Hint: Use v1.dot(v2) and (v1.dot(v1))**0.5 (no math import).
    4. Sort members by similarity (descending) and return the first n 'word' values.
    """
    pass

# TODO: Perform a loop to calculate the Silhouette score for k in [5, 8, 12, 16, 20].
# TODO: Find the k with the maximum score (best_k).
# TODO: Fit the final KMeans model using best_k.
# TODO: Use cluster_label to generate words for each cluster and show them in a Spark DataFrame.


### 7.3 Word2Vec + PCA inside a pipeline

In Section 7.2.1 you ranked the four text columns by standalone AUC. Use that result here: rather than concatenating all four columns blindly, choose which to include based on what you found.

**The dimensionality problem.** With `vectorSize=150`, the Word2Vec block contributes 150 dimensions to the final feature vector while the entire numeric and boolean block contributes roughly 20. A Random Forest sees far more candidate split points in the text block, which can crowd out the structured features even if they carry more signal.

**The fix: PCA.** We add a `PCA` stage immediately after `Word2Vec` to compress the 150-dimensional embedding down to `k` principal components — retaining most of the variance while bringing the text block back to a comparable scale. `PCA` is an estimator in `pyspark.ml.feature`: it fits on the training data to learn the principal directions, then transforms both splits.

**TODO:**
1. Choose `best_text_col` from your Section 7.2.1 ranking.
2. Set `k` — the number of PCA components to keep. Start with 20 and experiment.
3. Assemble the pipeline stages and assign to `word2vec_pipeline`.
4. Fit, transform, and compare AUC to the numeric-only baseline.

In [ ]:
from pyspark.ml.feature import PCA

# TODO: choose which text column(s) to use based on Section 7.2.1 results
best_text_col = # ...

# Tokenise and embed
tokenizer_stage = Tokenizer(inputCol=best_text_col, outputCol="text_tokens")

word2vec_stage = Word2Vec(
    vectorSize = 150,
    minCount   = 3,
    inputCol   = "text_tokens",
    outputCol  = "text_embedding",
    seed       = 753,
)

# PCA: compress 150-dim embedding to k components
# TODO: set k — try values between 10 and 50
pca_stage = PCA(
    k        = # ...,
    inputCol = "text_embedding",
    outputCol = "text_pca",
)

# Final assembly: text_pca is now comparable in size to the structured features
feature_assembler = VectorAssembler(
    inputCols = ["numeric_scaled"] + bool_cols + ["phenomenon_vec", "ghost_vec", "text_pca"],
    outputCol = "features",
)

rf = RandomForestClassifier(
    featuresCol = "features",
    labelCol    = TARGET,
    numTrees    = 100,
    maxDepth    = 5,
    seed        = 753,
)

# TODO: assemble into a Pipeline — stages in order:
# indexer, encoder, imputer, numeric_assembler, scaler,
# tokenizer_stage, word2vec_stage, pca_stage, feature_assembler, rf
word2vec_pipeline = # ...

train_text, test_text = [df.cache() for df in data_text.randomSplit([0.7, 0.3], seed=753)]
train_text.count(); test_text.count()

# TODO: fit, transform, evaluate
word2vec_pipeline_model = # ...
results_w2v = # ...
auc_w2v = evaluator.evaluate(results_w2v)
print(f"Word2Vec + PCA + RF — Test AUC-ROC: {auc_w2v:.4f}")
print(f"Numeric-only baseline (Section 4.3):  {auc_pipeline:.4f}")
print(f"Improvement: {auc_w2v - auc_pipeline:+.4f}")

In [ ]:


# # TODO: populate with (model_name, auc_score) for every pipeline you trained
pipeline_results = []

# --- plotting boilerplate — run once pipeline_results is populated ---
summary_df = spark.createDataFrame(pipeline_results, ["Model", "AUC_ROC"]) \
    .orderBy("AUC_ROC", ascending=False)
summary_df.show(truncate=False)

names  = [r["Model"]   for r in summary_df.collect()]
scores = [r["AUC_ROC"] for r in summary_df.collect()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(names, scores)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("AUC-ROC")
ax.set_title("Pipeline Comparison — HMSPIA NAPR")
plt.tight_layout()dsfsdsdfsdfsdfsdfsdfssdfsdfserwerwe
plt.show()